# Notebook 0: TCGA-BRCA pipeline (held-out test, repeated nested CV, PAM50 exclusion, stability, SHAP interpretation)
**Paper:** SHAP-Guided Selection of Gene Panel with Repeated Nested Cross-Validation and PAM50-Exclusion Benchmarking for Interpretable Breast Cancer Subtype Classification from RNA-Seq, with External Validation on METABRIC
**Authors:** Neetu Ahlawat, Vishal Verma (Chaudhary Ranbir Singh University, Jind, India)

| Section | Analysis | Paper |
|---|---|---|
| 1 | Data loading (514 tumours, 20,530 genes) and patient-level duplicate check | Section 3.1 |
| 2 | TreeSHAP selection (Algorithm 1), comparator selectors and classifiers | Sections 3.2–3.4; Table 2 |
| 3 | Held-out test (411 training / 103 test tumours; scaler and panel fitted on the training set only) | Section 4.1 |
| 4 | Repeated nested cross-validation (5 × 5 = 25 outer folds) for eight feature sets | Sections 3.5, 4.5 |
| 5 | Summary tables over outer folds | Tables 6–7 |
| 6 | Corrected resampled t-test (Nadeau and Bengio 2003) with Holm adjustment | Table 8 |
| 7 | Panel stability (Jaccard index) and PAM50 overlap | Table 9 |
| 8 | Accuracy by feature set and classifier | Fig. 6 |
| 9 | Held-out per-class metrics, confusion matrices, ROC curves, SHAP interpretation, learning curve and per-fold results | Sections 4.2–4.6; Tables 3–5, 10–11; Figs. 1–5, 7–8 |

**Data:** upload `HiSeqV2` and `TCGA.BRCA.sampleMap_BRCA_clinicalMatrix` from UCSC Xena (https://xenabrowser.net) to `/content/`.

**Runtime:** choose *Runtime → Change runtime type → T4 GPU*. Section 4 takes about 95 minutes on a T4 GPU (set `N_REPEATS = 1` for a quick test of about 20 minutes). TreeSHAP values are taken from XGBoost's exact `pred_contribs=True` routine. Results are written to `OUT_DIR`.

## 0. Setup and configuration

In [ ]:
import os, json, time, itertools, warnings
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

import xgboost as xgb

# ---------------- configuration ----------------
EXPR_PATH = "/content/HiSeqV2"                                   # UCSC Xena, log2(RSEM+1)
CLINICAL_PATH = "/content/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix"  # UCSC Xena clinical matrix
OUT_DIR = "/content/PAM50_results_top300"                         # or a Google Drive folder
N_REPEATS = 5          # 5 repeats x 5 folds = 25 outer folds, as in the paper (1 = quick test)
USE_GPU = True         # needs a GPU runtime; set False on CPU
INCLUDE_ALL = True     # all-gene ablation (slowest part)
RUN_MI = False
TOP_N = 300
CLASS_ORDER = ["Basal-like", "HER2-enriched", "Luminal A", "Luminal B"]
SUBTYPE_COL_CANDIDATES = ["PAM50_mRNA_nature2012", "PAM50Call_RNAseq", "PAM50_mRNA"]
# The 50 PAM50 genes (some symbols differ between annotation versions; the notebook reports which are found)
PAM50 = ["ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20","CDC6","CDH3","CENPF","CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1","FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C","KRT14","KRT17","KRT5","MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC","NAT1","NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"]
PAM50_ALIASES = {"ORC6":"ORC6L","NDC80":"KNTC2","NUF2":"CDCA1"}

os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(OUT_DIR)

## 1. Data loading

In [ ]:
# ---------------- data ----------------
import os


expr_raw = pd.read_csv(EXPR_PATH, sep="\t", index_col=0)
expr = expr_raw.T if expr_raw.shape[0] > expr_raw.shape[1] else expr_raw.copy()
expr.index = expr.index.astype(str).str[:15]
clinical = pd.read_csv(CLINICAL_PATH, sep="\t", index_col=0, low_memory=False)
clinical.index = clinical.index.astype(str)
subtype_col = next(c for c in SUBTYPE_COL_CANDIDATES if c in clinical.columns)
cs = clinical[[subtype_col]].dropna(); cs.columns = ["subtype"]; cs.index = cs.index.str[:15]
merged = expr.join(cs, how="inner")
merged = merged[merged["subtype"].isin(["Basal-like","HER2-enriched","LumA","LumB","Luminal A","Luminal B"])]
merged["subtype"] = merged["subtype"].replace({"LumA":"Luminal A","LumB":"Luminal B"})
le = LabelEncoder(); le.fit(CLASS_ORDER); y = le.transform(merged["subtype"])
X_df = merged.drop(columns="subtype").apply(pd.to_numeric, errors="coerce").fillna(0)
gene_names = np.array(X_df.columns.tolist()); X = X_df.values
patient_ids = pd.Series(X_df.index)
print("Data:", X.shape, "class counts:", np.bincount(y))

pam_found = []
for g in PAM50:
    gg = g if g in set(gene_names) else PAM50_ALIASES.get(g)
    if gg in set(gene_names): pam_found.append(gg)
pam_idx = np.array([int(np.where(gene_names == g)[0][0]) for g in pam_found])
print(f"PAM50 genes found in the matrix: {len(pam_found)} of {len(PAM50)}")
assert X.shape[1] > 20000, f"Only {X.shape[1]} genes loaded; the HiSeqV2 file may be incomplete"
assert len(pam_found) >= 45, f"Only {len(pam_found)} PAM50 genes found"

### Patient-level check (is any patient present more than once?)

In [ ]:
pid12 = patient_ids.str[:12]
dups = pid12[pid12.duplicated(keep=False)]
print("Samples:", len(pid12), "| unique patients:", pid12.nunique(), "| samples sharing a patient ID:", len(dups))
if len(dups): print(dups.value_counts().head(10))

## 2. Selection and model functions (settings of Table 2)

In [ ]:
def xgb_params(seed):
    p = dict(objective="multi:softprob", num_class=4, max_depth=5, eta=0.05, tree_method="hist",
             seed=seed, eval_metric="mlogloss", nthread=-1)
    if USE_GPU: p["device"] = "cuda"
    return p

def importances_xgb(Xtr, ytr, seed, exclude_idx=None):
    """Return (mean |SHAP| over samples and classes, gain importance) from one XGBoost fit.
    TreeSHAP values come from XGBoost's pred_contribs=True (exact TreeSHAP)."""
    p = Xtr.shape[1]
    keep = np.arange(p) if exclude_idx is None else np.setdiff1d(np.arange(p), exclude_idx)
    d = xgb.DMatrix(Xtr[:, keep], label=ytr)
    bst = xgb.train(xgb_params(seed), d, num_boost_round=300)
    c = bst.predict(d, pred_contribs=True)
    if c.ndim == 2: c = c.reshape(len(ytr), 4, -1)
    shap_k = np.abs(c[:, :, :-1]).mean(axis=(0, 1))          # Algorithm 1: mean |SHAP| over samples and classes
    sc = bst.get_score(importance_type="gain"); gain_k = np.zeros(len(keep))
    for k, v in sc.items(): gain_k[int(k[1:])] = v
    shap_imp = np.full(p, -1.0); gain_imp = np.full(p, -1.0)
    shap_imp[keep] = shap_k; gain_imp[keep] = gain_k
    return shap_imp, gain_imp

def top_k(imp, k=None): return np.argsort(-imp, kind="stable")[: (k or TOP_N)]

def l1_importance(Xtr, ytr, seed, anova_F):
    ovr = OneVsRestClassifier(LogisticRegression(penalty="l1", solver="liblinear", C=0.5, max_iter=300, random_state=seed)).fit(Xtr, ytr)
    imp = np.abs(np.vstack([e.coef_ for e in ovr.estimators_])).mean(axis=0)
    return imp + 1e-9 * (anova_F / (np.nanmax(anova_F) + 1e-12))   # ANOVA F breaks ties among zero coefficients

def make_models(seed):
    m = {"XGBoost": xgb.XGBClassifier(objective="multi:softprob", num_class=4, n_estimators=300, max_depth=5,
                                      learning_rate=0.05, random_state=seed, eval_metric="mlogloss", n_jobs=-1,
                                      **({"device": "cuda"} if USE_GPU else {}))}
    m.update({"Random Forest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
              "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=seed),
              "Logistic Regression": LogisticRegression(max_iter=5000, random_state=seed)})
    return m

def metrics(model, Xte, yte):
    pred = model.predict(Xte); proba = model.predict_proba(Xte); yb = label_binarize(yte, classes=[0, 1, 2, 3])
    return dict(accuracy=accuracy_score(yte, pred) * 100, balanced_accuracy=balanced_accuracy_score(yte, pred) * 100,
                f1_macro=f1_score(yte, pred, average="macro") * 100, f1_weighted=f1_score(yte, pred, average="weighted") * 100,
                auc_weighted=roc_auc_score(yb, proba, average="weighted") * 100, auc_macro=roc_auc_score(yb, proba, average="macro") * 100)

## 3. Held-out test (scaler and panel fitted on the 411 training tumours only)

In [ ]:
SEED0 = 42
Xtr_raw, Xte_raw, ytr_, yte_ = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED0)
sc = StandardScaler().fit(Xtr_raw); Xtr_, Xte_ = sc.transform(Xtr_raw), sc.transform(Xte_raw)
t0 = time.time(); shap_imp, gain_imp = importances_xgb(Xtr_, ytr_, SEED0); sel = top_k(shap_imp)
print(f"selection took {time.time()-t0:.0f}s; first 20 genes:", [str(g) for g in gene_names[sel][:20]])
rows = []
for name, m in make_models(SEED0).items():
    m.fit(Xtr_[:, sel], ytr_); r = metrics(m, Xte_[:, sel], yte_); r["model"] = name; rows.append(r)
held = pd.DataFrame(rows).set_index("model").round(2); print(held); held.to_csv("heldout_train_only_scaler.csv")

In [ ]:
# Fig. 2: held-out performance of the four classifiers across the five aggregate metrics
import matplotlib.pyplot as plt
mets = ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted", "auc_weighted"]
labels = ["Accuracy", "Balanced accuracy", "Macro F1", "Weighted F1", "Weighted ROC-AUC"]
fig, ax = plt.subplots(figsize=(9, 4)); w = 0.2
for j, mdl in enumerate(held.index):
    ax.bar(np.arange(len(mets)) + j * w, held.loc[mdl, mets], w, label=mdl)
ax.set_xticks(np.arange(len(mets)) + 1.5 * w); ax.set_xticklabels(labels)
ax.set_ylabel("Score (%)"); ax.set_ylim(70, 100); ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.savefig("fig2_heldout_metrics.png", dpi=300); plt.show()

## 4. Repeated nested cross-validation (selection and scaling inside every outer fold)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif

methods = ["SHAP300", "GAIN300", "ANOVA300", "L1_300", "RANDOM300", "PAM50", "SHAP300_noPAM50"] + (["MI300"] if RUN_MI else []) + (["ALL"] if INCLUDE_ALL else [])
records, panels = [], {}
t_start = time.time()
for r in range(N_REPEATS):
    seed = SEED0 + r
    for f, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)):
        sc = StandardScaler().fit(X[tr]); Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te]); ytr, yte = y[tr], y[te]
        shap_imp, gain_imp = importances_xgb(Xtr, ytr, seed)
        F, _ = f_classif(Xtr, ytr); F = np.nan_to_num(F)
        sel = {"SHAP300": top_k(shap_imp), "GAIN300": top_k(gain_imp), "ANOVA300": top_k(F),
               "L1_300": top_k(l1_importance(Xtr, ytr, seed, F)),
               "RANDOM300": np.random.default_rng(seed * 100 + f).choice(X.shape[1], TOP_N, replace=False),
               "PAM50": pam_idx}
        s2, _ = importances_xgb(Xtr, ytr, seed, exclude_idx=pam_idx); sel["SHAP300_noPAM50"] = top_k(s2)
        if RUN_MI: sel["MI300"] = top_k(mutual_info_classif(Xtr, ytr, n_neighbors=3, random_state=seed))
        if INCLUDE_ALL: sel["ALL"] = np.arange(X.shape[1])
        for mth, idx in sel.items():
            if mth != "ALL": panels[f"{mth}|{r}|{f}"] = [str(g) for g in gene_names[idx]]
            for name, model in make_models(seed).items():
                model.fit(Xtr[:, idx], ytr); rec = metrics(model, Xte[:, idx], yte)
                rec.update(repeat=r, fold=f, selection=mth, model=name); records.append(rec)
        print(f"repeat {r+1}/{N_REPEATS} fold {f+1}/5 done, elapsed {(time.time()-t_start)/60:.1f} min", flush=True)
    pd.DataFrame(records).to_csv("nested_results_long.csv", index=False); json.dump(panels, open("panels.json", "w"))
df = pd.DataFrame(records); print("saved nested_results_long.csv and panels.json")

## 5. Summary tables (Tables 6–7)

In [ ]:
M = ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted", "auc_weighted"]
unit = df.groupby(["selection", "model"])[M].agg(["mean", "std"]).round(2)
print("Mean and SD over all outer folds (repeat x fold):"); print(unit.to_string())
rep = df.groupby(["selection", "model", "repeat"])[M].mean().groupby(["selection", "model"]).agg(["mean", "std"]).round(2)
unit.to_csv("summary_over_outer_folds.csv"); rep.to_csv("summary_over_repeats.csv")

## 6. Corrected resampled t-test: SHAP-300 versus every other feature set (Holm-adjusted)

In [ ]:
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p, dtype=float); run = 0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1, run)
    return adj
n_te, n_tr = len(y) / 5, len(y) * 4 / 5; out = []
for model in df["model"].unique():
    base = df[(df.selection == "SHAP300") & (df.model == model)].sort_values(["repeat", "fold"])["accuracy"].values
    rows = []
    for mth in [m for m in methods if m != "SHAP300"]:
        other = df[(df.selection == mth) & (df.model == model)].sort_values(["repeat", "fold"])["accuracy"].values
        d = base - other; N = len(d)
        if N < 3 or d.std(ddof=1) == 0: continue
        t = d.mean() / np.sqrt((1 / N + n_te / n_tr) * d.var(ddof=1)); p = 2 * stats.t.sf(abs(t), N - 1)
        rows.append(dict(model=model, comparison=f"SHAP300 - {mth}", mean_diff=round(d.mean(), 2), t=round(t, 2), p=p))
    if rows:
        adj = holm([r["p"] for r in rows])
        for r_, a in zip(rows, adj): r_["p_holm"] = round(a, 4); r_["p"] = round(r_["p"], 4); out.append(r_)
ttest = pd.DataFrame(out); print(ttest.to_string(index=False)) if len(ttest) else print("Need at least 3 outer folds (N_REPEATS >= 1 gives N = 5)")
ttest.to_csv("corrected_ttest.csv", index=False)

## 7. Panel stability (Jaccard index) and overlap with PAM50 genes

In [ ]:
def jac(a, b): a, b = set(a), set(b); return len(a & b) / len(a | b)
st = []
for mth in [m for m in methods if m not in ("ALL", "PAM50")]:
    for r in range(N_REPEATS):
        P = [panels[f"{mth}|{r}|{f}"] for f in range(5)]
        pj = [jac(a, b) for a, b in itertools.combinations(P, 2)]
        st.append(dict(selection=mth, repeat=r, mean_jaccard=np.mean(pj), genes_in_all_5_folds=len(set.intersection(*map(set, P))),
                       genes_in_ge4_folds=sum(1 for g in set.union(*map(set, P)) if sum(g in p for p in P) >= 4),
                       mean_pam50_genes_in_panel=np.mean([len(set(p) & set(pam_found)) for p in P])))
stab = pd.DataFrame(st).groupby("selection").mean(numeric_only=True).round(3).drop(columns="repeat"); print(stab); stab.to_csv("panel_stability.csv")

## 8. Accuracy by feature set and classifier (Fig. 6)

In [ ]:
import matplotlib.pyplot as plt
piv = df.groupby(["selection", "model"])["accuracy"].agg(["mean", "std"]).reset_index()
fig, ax = plt.subplots(figsize=(11, 4.5)); w = 0.2
for i, mdl in enumerate(df["model"].unique()):
    s = piv[piv.model == mdl].set_index("selection").reindex(methods)
    ax.bar(np.arange(len(methods)) + i * w, s["mean"], w, yerr=s["std"], capsize=2, label=mdl)
ax.set_xticks(np.arange(len(methods)) + 1.5 * w); ax.set_xticklabels(methods, rotation=30, ha="right"); ax.set_ylabel("Nested-CV accuracy (%)")
ax.set_ylim(60, 100); ax.legend(ncol=4, fontsize=8); ax.set_title("Accuracy by selection method (mean and SD over outer folds)")
plt.tight_layout(); plt.savefig("fig_selection_comparison.png", dpi=300); plt.show()
print("Results saved in", OUT_DIR)

## 9. Held-out per-class metrics, confusion matrices, ROC curves, SHAP interpretation, learning curve and per-fold results

In [ ]:
# Rebuilds the held-out panel from the 411 training tumours only (independent of the nested loop above).
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import learning_curve

Xtr_raw, Xte_raw, ytr_, yte_ = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED0)
sc = StandardScaler().fit(Xtr_raw); Xtr_, Xte_ = sc.transform(Xtr_raw), sc.transform(Xte_raw)
shap_imp_h, _ = importances_xgb(Xtr_, ytr_, SEED0)
heldout_sel = top_k(shap_imp_h)
print("Training-only panel, first 20:", [str(g) for g in gene_names[heldout_sel][:20]])
assert len(set(heldout_sel)) == TOP_N

yb = label_binarize(yte_, classes=[0, 1, 2, 3])
fitted, report_rows, auc_rows, summary_rows, cms = {}, [], [], [], {}
for name, m in make_models(SEED0).items():
    m.fit(Xtr_[:, heldout_sel], ytr_)
    pred = m.predict(Xte_[:, heldout_sel]); proba = m.predict_proba(Xte_[:, heldout_sel]); fitted[name] = (m, proba)
    rep = classification_report(yte_, pred, target_names=CLASS_ORDER, digits=3, output_dict=True)
    print("=====", name, "accuracy", round(accuracy_score(yte_, pred) * 100, 2))
    print(classification_report(yte_, pred, target_names=CLASS_ORDER, digits=2))
    cms[name] = confusion_matrix(yte_, pred); print(cms[name])
    aucs = [roc_auc_score(yb[:, k], proba[:, k]) for k in range(4)]
    print("per-class AUC:", [round(float(a), 3) for a in aucs])
    for k, c in enumerate(CLASS_ORDER):
        report_rows.append(dict(model=name, subtype=c, precision=rep[c]["precision"], recall=rep[c]["recall"], f1=rep[c]["f1-score"], support=rep[c]["support"], auc=aucs[k]))
    summary_rows.append(dict(model=name, accuracy=accuracy_score(yte_, pred) * 100))
pd.DataFrame(report_rows).round(3).to_csv("heldout_per_class_clean.csv", index=False)
print("\nCheck: these accuracies should equal those of Section 3:")
print(pd.DataFrame(summary_rows).round(2).to_string(index=False)); print(held[["accuracy"]])

# Fig. 3 confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (name, cm) in zip(axes, cms.items()):
    ax.imshow(cm, cmap="Blues"); ax.set_title(name); ax.set_xticks(range(4), CLASS_ORDER, rotation=35, ha="right", fontsize=8); ax.set_yticks(range(4), CLASS_ORDER, fontsize=8)
    for i in range(4):
        for j in range(4): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("fig3_confusion_clean.png", dpi=300); plt.show()

# Fig. 4 ROC curves
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (name, (m, proba)) in zip(axes, fitted.items()):
    for k, c in enumerate(CLASS_ORDER):
        fpr, tpr, _ = roc_curve(yb[:, k], proba[:, k]); ax.plot(fpr, tpr, label=f"{c} ({roc_auc_score(yb[:, k], proba[:, k]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.7); ax.set_title(name); ax.set_xlabel("False positive rate"); ax.legend(fontsize=7)
axes[0].set_ylabel("True positive rate"); plt.tight_layout(); plt.savefig("fig4_roc_clean.png", dpi=300); plt.show()

# Top 20 SHAP genes on the test set, and Figs. 7 and 8
xgbm = fitted["XGBoost"][0]
c = xgbm.get_booster().predict(xgb.DMatrix(Xte_[:, heldout_sel]), pred_contribs=True).reshape(len(yte_), 4, -1)
imp = np.abs(c[:, :, :-1]).mean(axis=(0, 1)); top = np.argsort(-imp)[:20]
top_tbl = pd.DataFrame({"rank": range(1, 21), "gene": gene_names[heldout_sel][top], "mean_abs_shap": imp[top].round(6)})
print(top_tbl.to_string(index=False)); top_tbl.to_csv("heldout_top20_shap_clean.csv", index=False)
per_sample = np.abs(c[:, :, :-1]).mean(axis=1)
fig, ax = plt.subplots(figsize=(8, 6)); rng = np.random.default_rng(0)
for pos, g in enumerate(top[:15][::-1]):
    v = Xte_[:, heldout_sel][:, g]; pts = ax.scatter(per_sample[:, g], pos + rng.uniform(-.25, .25, len(v)), c=np.clip(v, -2, 2), cmap="coolwarm", s=10)
ax.set_yticks(range(15), gene_names[heldout_sel][top[:15][::-1]]); ax.set_xlabel("Absolute SHAP value per sample"); plt.colorbar(pts, label="Standardised expression")
plt.tight_layout(); plt.savefig("fig7_shap_distribution_clean.png", dpi=300); plt.show()
fig, ax = plt.subplots(figsize=(7, 6)); ax.barh(top_tbl.gene[::-1], top_tbl.mean_abs_shap[::-1], color="#1f77b4"); ax.set_xlabel("Mean absolute SHAP value")
plt.tight_layout(); plt.savefig("fig8_shap_importance_clean.png", dpi=300); plt.show()

# Fig. 5 learning curve inside the training partition
sizes, tr_s, va_s = learning_curve(make_models(SEED0)["XGBoost"], Xtr_[:, heldout_sel], ytr_, cv=StratifiedKFold(5, shuffle=True, random_state=SEED0), train_sizes=np.linspace(.2, 1, 5), n_jobs=1)
lc = pd.DataFrame({"n": sizes, "train": tr_s.mean(1) * 100, "validation": va_s.mean(1) * 100, "validation_sd": va_s.std(1) * 100}).round(2); print(lc.to_string(index=False)); lc.to_csv("learning_curve_clean.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4)); ax.plot(lc.n, lc.train, marker="o", label="Training"); ax.plot(lc.n, lc.validation, marker="o", label="Validation")
ax.fill_between(lc.n, lc.validation - lc.validation_sd, lc.validation + lc.validation_sd, alpha=.2); ax.set_xlabel("Training tumours"); ax.set_ylabel("Accuracy (%)"); ax.legend()
plt.tight_layout(); plt.savefig("fig5_learning_curve_clean.png", dpi=300); plt.show()

# Per-fold table and SHAP300 accuracy range, from the nested results
d0 = pd.read_csv("nested_results_long.csv"); s = d0[d0.selection == "SHAP300"]
print(s[s.repeat == 0].pivot(index="model", columns="fold", values="accuracy").round(2))
print(s.groupby("model")["accuracy"].agg(["min", "max"]).round(2))